In [0]:
# Cell 1
import os, sys, uuid
sys.path.insert(0, os.path.abspath(".."))

from ecom.audit import Audit
from ecom.gold import run_gold

dbutils.widgets.text("catalog", "ecom_dev")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("bundle_target", "dev")

catalog = dbutils.widgets.get("catalog")
run_id = dbutils.widgets.get("run_id") or str(uuid.uuid4())
bundle_target = dbutils.widgets.get("bundle_target")
triggered_by = spark.sql("SELECT current_user()").first()[0]
print(f"catalog={catalog} run_id={run_id}")

In [0]:
# Cell 2
audit = Audit(spark, catalog, run_id, "gold_transform", bundle_target, triggered_by)
run_gold(spark, catalog, run_id, audit)

In [0]:
# Cell 3
display(spark.sql(f"""
    SELECT table_name, status, rows_read, rows_written, target_version,
           round(duration_sec, 1) AS seconds, error_message
    FROM {catalog}.control.pipeline_run_audit
    WHERE run_id = '{run_id}' AND layer = 'gold'
    ORDER BY table_name
"""))